<h1 style="text-align: center">Notebook 0: Introducción a MONAI</h1>
<h2 style="text-align: center">Parte 1</h2>

<p style="text-align: center">Unidad 1: Fundamentos de IA</p>
<p style="text-align: center">Este cuaderno nos introduce en MONAI Core. Veremos algunos ejemplos prácticos sobre transformaciones, cargadores de datos, caché y redes.</p>

### Comprobar el acceso a la GPU

Si ejecutan **!nvidia-smi** en una celda, podrán comprobar el tipo de hardware al que tiene acceso.

In [ ]:
!nvidia-smi

### Paquetes necesarios para ejecutar en Colab

Ejecutar la siguiente celda para instalar MONAI la primera vez que se ejecute este notebook en Colab:

In [ ]:
!python -c "import monai" || pip install -qU "monai[ignite, nibabel, torchvision, tqdm]"
!pip install pydicom

## Primeros pasos con MONAI

MONAI es un marco de código abierto basado en PyTorch para el aprendizaje profundo en el ámbito de las imágenes médicas, que forma parte del ecosistema de PyTorch.


### ¿Por qué un marco específico?

* Las aplicaciones biomédicas tienen requisitos específicos.
* Las modalidades de imagen (RM, TC, ecografía, etc.) requieren funcionalidades específicas de procesamiento de datos.
* Los formatos de datos (DICOM, NIfTI, etc.) son específicos de las aplicaciones médicas y requieren un soporte concreto.
* Ciertas arquitecturas de red están diseñadas para aplicaciones biomédicas o son muy adecuadas para ellas.
* Las transformaciones de datos específicas de las aplicaciones biomédicas y de las modalidades de imagen resultan muy útiles para preprocesar datos, aumentar el volumen de datos durante el entrenamiento y para el posprocesamiento.
* La ciencia reproducible requiere experimentos reproducibles que, a su vez, dependen de software accesible para otros científicos, aunque solo sea como punto de partida común.


### ¿Por qué MONAI?

MONAI proporciona un marco de funcionalidades e infraestructura de aprendizaje profundo para satisfacer estas necesidades de forma flexible y compatible con PyTorch:

* Compatibilidad directa para cargar y manipular tipos de archivos biomédicos.
* Transformaciones específicas para datos biomédicos destinadas a la regularización y al aumento de datos de imágenes biomédicas para el entrenamiento, la validación y la implementación.
* Biblioteca de definiciones de redes, métricas y funciones de pérdida de uso general que implementan tanto arquitecturas consolidadas como de vanguardia.
* Conjunto de componentes listos para usar para el entrenamiento y la inferencia, con el fin de utilizar la infraestructura informática de manera eficiente.



## Objetivos de aprendizaje

Para comprender mejor las transformaciones de MONAI, el caché de datasets y las arquitecturas de redes, este notebook nos ayudará a responder seis preguntas clave:

1. **¿Qué transformaciones están disponibles para crear una pipeline de datos para el entrenamiento?**
2. **¿Qué se necesita para escribir una transformación personalizada?**
3. **¿Cómo creo un dataset básico de MONAI con transformaciones?**
4. **¿Qué es un dataset de MONAI y cómo funciona el caché de datasets?**
5. **¿Qué datasets comunes proporciona MONAI?**
6. **¿Qué redes y componentes de red proporciona MONAI y cómo se usan para crear una red?**

### Importaciones

Empecemos importando nuestras dependencias:
* Vamos a cargar todo lo que necesitaremos durante el resto del notebook.

### Importaciones

In [ ]:
import tempfile
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import time
import torch

import monai
from monai.config import print_config
from monai.utils import first
from monai.config import KeysCollection

from monai.data import (Dataset, ArrayDataset, create_test_image_3d, DataLoader, DataLoader, 
CacheDataset, PersistentDataset, SmartCacheDataset)

from monai.transforms import (Transform, MapTransform, Randomizable, EnsureChannelFirst, EnsureChannelFirstd,
Compose, LoadImage, LoadImaged, Lambda, Lambdad, RandSpatialCrop, RandSpatialCropd, RandGaussianNoise,
RandGaussianNoised, Orientation, Rotate, MapTransform)

from monai.apps import DecathlonDataset, TciaDataset
from monai.apps.tcia import TCIA_LABEL_DICT

### Verifique su configuración

Antes de empezar, puede ser útil verificar la configuración de su entorno y asegurarse de que todas las dependencias que necesita estén instaladas. MONAI proporciona una utilidad práctica para facilitarlo con `print_config()`.

In [ ]:
print_config()

### 1. ¿Qué transformaciones están disponibles para ayudar a crear un pipeline de datos para el entrenamiento?


#### Entrada/salida, procesamiento y aumento de datos de imágenes médicas

Las imágenes médicas requieren métodos muy especializados para I/O, preprocesamiento y aumento. Las imágenes médicas suelen estar en formatos especializados con metainformación rica y los volúmenes de datos suelen ser de alta dimensión. Todo ello requiere procedimientos de manipulación cuidadosamente diseñados. El enfoque en imágenes médicas de MONAI se habilita mediante transformaciones de imagen potentes y flexibles que facilitan pipelines de preprocesamiento de datos médicos fáciles de usar, reproducibles y optimizadas.

<img src="../imagenes/medical_transforms.png" />

#### Las transformaciones admiten datos en formato de diccionario y de array

Los paquetes de visión artificial más usados (como torchvision) se centran en el procesamiento espacial de imágenes en arrays 2D. MONAI proporciona transformaciones más específicas del dominio tanto para 2D como para 3D y conserva la característica flexible de “compose”.

Como el preprocesamiento de imágenes médicas suele requerir parámetros de sistema adicionales y muy precisos, MONAI proporciona transformaciones para datos de entrada encapsulados en diccionarios de Python. Los usuarios pueden especificar las claves correspondientes a los campos de datos esperados y a los parámetros del sistema para componer transformaciones complejas.

Hay un amplio conjunto de transformaciones en seis categorías: Crop & Pad, Intensity, IO, Post-processing, Spatial y Utilities. Para obtener más detalles, visite todas las transformaciones de MONAI.

#### Transformaciones específicas para medicina
MONAI tiene como objetivo proporcionar un conjunto completo de transformaciones específicas para imágenes médicas. Actualmente incluyen, por ejemplo:

* LoadImage: carga archivos en formatos específicos de imágenes médicas desde la ruta proporcionada.
* Spacing: remuestrea la imagen de entrada en el pixdim especificado.
* Orientation: cambia la orientación de la imagen a los axcodes especificados.
* RandGaussianNoise: perturba las intensidades de la imagen añadiendo ruido estadístico.
* NormalizeIntensity: normalización de intensidad basada en media y desviación estándar.
* Affine: transforma la imagen según los parámetros afines.
* Rand2DElastic: deformación elástica aleatoria y afín en 2D.
* Rand3DElastic: deformación elástica aleatoria y afín en 3D.

<img src="../imagenes/image_prop.svg" style="background-color: white; padding: 8px;" alt="Diagrama técnico sobre propiedades espaciales de una imagen médica."/>

#### Crear datos de ejemplo y un directorio temporal para los ejemplos

Crearemos un directorio temporal y lo rellenaremos con algunas imágenes de ejemplo en formato NIfTI que contienen una mezcla aleatoria de esferas. También estamos creando un par de segmentaciones coincidente que se utilizará más adelante en el cuaderno.

In [ ]:
fn_keys = ("img", "seg")  # filename keys for image and seg files

root_dir = tempfile.mkdtemp()
filenames = []

for i in range(5):
    im, seg = create_test_image_3d(128, 128, 128, num_objs=16, rad_max=25)

    im_filename = f"{root_dir}/im{i}.nii.gz"
    seg_filename = f"{root_dir}/seg{i}.nii.gz"
    filenames.append({"img": im_filename, "seg": seg_filename})

    n = nib.Nifti1Image(im, np.eye(4))
    nib.save(n, im_filename)

    n = nib.Nifti1Image(seg, np.eye(4))
    nib.save(n, seg_filename)

#### Transformaciones de array

Las transformaciones en MONAI son objetos invocables que aceptan entradas desde datos iniciales en un dataset o desde transformaciones previas. Podemos crearlas y llamarlas directamente sin ninguna infraestructura o configuración del sistema, ya que los componentes de MONAI están diseñados para ser lo más desacoplados posible. Por ejemplo, podemos cargar uno de nuestros archivos NIfTI directamente creando la transformación y llamándola.


:::{.callout-note title="Nota"}
En Python, un objeto invocable (o callable) es cualquier objeto que se puede llamar usando paréntesis (), como funciones, métodos o clases. Puedes hacer que tus propias clases creen objetos invocables definiendo el método especial .__call__() dentro de ellas.
:::

:::{.callout-note title="Nota"}
Pytorch define los tensores de la siguiente manera:
- **2D**: [B, C, H, W] donde B es el tamaño del batch, C es el número de canales, H es la altura y W es el ancho. Esto significa, por ejemplo, que cada imagen debe tener un canal (C=1) y que las imágenes deben ser de tamaño 28x28 (H=28, W=28). MONAI proporciona transformaciones para asegurarse de que las imágenes tengan el formato correcto.

- **3D**: [B, C, D, H, W] donde B es el tamaño del batch, C es el número de canales, D es la profundidad, H es la altura y W es el ancho. Esto significa, por ejeemplo, que cada imagen debe tener un canal (C=1) y que las imágenes deben ser de tamaño 64x64x64 (D=64, H=64, W=64). MONAI proporciona transformaciones para asegurarse de que las imágenes tengan el formato correcto.
:::

In [ ]:
trans = Compose([LoadImage(image_only=True), EnsureChannelFirst()])
img = trans(filenames[0]["img"])
print(type(img), img.shape, img.get_device())

#### Transformaciones de diccionario

Hasta ahora hemos visto transformaciones aplicadas a arrays individuales de NumPy; sin embargo, para la mayoría de los esquemas de entrenamiento se necesita una canalización con múltiples valores:
* Para ello, MONAI incluye transformaciones para operar sobre diccionarios de arrays, una por cada transformación equivalente de array.
* Estas pueden aplicarse a valores con nombre en un diccionario de entrada mientras se dejan sin tocar los valores sin nombre.
  * por ejemplo, añadir ruido a una imagen sin modificar la imagen de etiqueta asociada.

#### Transformaciones de diccionario

Más arriba en el cuaderno importamos las transformaciones equivalentes de diccionario, que tienen una 'd' añadida a sus nombres:
* Usaremos esas transformaciones en esta sección.
* El argumento keys en LoadImaged se usa para indicar qué claves contienen rutas a archivos NIfTI.
  * Todos los demás valores del diccionario de entrada se conservarán.
  * Con esto podemos ver las claves devueltas al llamar a la transformación:

In [ ]:
trans = LoadImaged(keys=fn_keys)
data = trans(filenames[0])
print(list(data.keys()))

### ¿Qué se necesita para escribir una transformación personalizada?



#### Transformación personalizada de array

Podemos definir nuestra propia operación de transformación personalizada de varias maneras.

Si se usa un callable simple como operador, `Lambda` se puede usar para envolverlo como una transformación:
* Definimos en este ejemplo una transformación para sumar la imagen en la primera dimensión (ancho) y obtener una imagen 2D:

In [ ]:
def sum_width(img):
    return img.sum(1)


trans = Compose([LoadImage(image_only=True), EnsureChannelFirst(), Lambda(sum_width)])
img = trans(filenames[0]["img"])
plt.imshow(img[0])

Crear una subclase de `Transform` es el segundo método, y tiene la ventaja de poder definir atributos con los objetos instanciados:
* Definamos una clase para sumar en una dimensión elegida y usemos la segunda dimensión (altura):

In [ ]:
class SumDimension(Transform):
    def __init__(self, dim=1):
        self.dim = dim

    def __call__(self, inputs):
        return inputs.sum(self.dim)


trans = Compose([LoadImage(image_only=True), EnsureChannelFirst(), SumDimension(2)])
img = trans(filenames[0]["img"])
plt.imshow(img[0])

#### Transformación personalizada de diccionario

`Lambdad` aplica la función dada a cada array nombrado por `keys` por separado:
* Podemos usar esto para definir transformaciones que actúen sobre distintos valores con nombre del diccionario en diferentes puntos de la secuencia:

In [ ]:
def sum_width(img):
    return img.sum(1)

def max_width(img):
    return img.max(1)

trans = Compose(
    [LoadImaged(fn_keys), EnsureChannelFirstd(fn_keys), Lambdad(("img",), sum_width), Lambdad(("seg",), max_width)]
)

imgd = trans(filenames[0])
img = imgd["img"]
seg = imgd["seg"]

plt.imshow(np.hstack((img[0] * 5 / img.max(), np.squeeze(seg[0]))))

#### Transformaciones deterministas frente a no deterministas

::: {.callout-tip}
## *Transformaciones deterministas*

Dado el mismo input, producen el mismo resultado.
:::

::: {.callout-tip}
## *Transformaciones no deterministas*

Dado el mismo input, producen un resultado aleatorio cada vez, basado en el RNG.
:::

Las transformaciones no deterministas de MONAI tienen varias funciones útiles:
* Por defecto, cada transformación tiene su propio RNG.
* Los datasets pueden sobrescribir esto (véase `ArrayDataset` más abajo).
* Se puede sobrescribir proporcionando una semilla ('seed') o una instancia de `RandomState`.


### 3. ¿Cómo creo un dataset básico de MONAI con transformaciones?

Ahora que hemos visto las transformaciones, echemos un vistazo a los datasets:
* Con una fuente de datos y transformaciones definidas, ya podemos crear un objeto dataset.
* La clase base de MONAI es `Dataset`, creada aquí para cargar únicamente los archivos de imagen NIfTI.
* `Dataset` hereda de la clase de PyTorch con ese nombre y añade únicamente la capacidad de aplicar la transformación dada a los elementos seleccionados.
  * Si está familiarizado con la clase de PyTorch, esto funcionará de la misma manera.

In [ ]:
images = [fn["img"] for fn in filenames]

transform = Compose([LoadImage(image_only=True), EnsureChannelFirst()])
ds = Dataset(images, transform)
img_tensor = ds[0]
print(img_tensor.shape, img_tensor.get_device())

#### ArrayDataset

MONAI proporciona `ArrayDataset` para aplicaciones de entrenamiento supervisado:
* Proporciona una canalización para imágenes y otra para etiquetas.
* Normalmente esto es útil al realizar aumentos en imágenes que no tienen sentido para las etiquetas.
  * Añadir ruido, normalizar, etc.
* Ambas canalizaciones usan el estado aleatorio del dataset para garantizar la coherencia en las operaciones aleatorias.
  * Precaución: las operaciones compartidas deben estar en el mismo orden.
  * Precaución: las operaciones compartidas deben ir antes que cualquier operación no compartida.

#### ArrayDataset

In [ ]:
images = [fn["img"] for fn in filenames]
segs = [fn["seg"] for fn in filenames]

img_transform = Compose(
    [
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        RandSpatialCrop((128, 128, 128), random_size=False),
        RandGaussianNoise(.5, 1, 1),
    ]
)
seg_transform = Compose(
    [LoadImage(image_only=True), EnsureChannelFirst(), RandSpatialCrop((128, 128, 128), random_size=False)]
)

ds = ArrayDataset(images, img_transform, segs, seg_transform)
im, seg = ds[0]
plt.imshow(np.hstack([im.numpy()[0, 48], seg.numpy()[0, 48]]))

#### Dataset con transformaciones basadas en diccionarios

Alternativamente, `Dataset` se puede usar con transformaciones basadas en diccionarios para construir un mapeo de resultados. Para aplicaciones de entrenamiento más allá de pares simples de entrada/verdad básica como los anteriores, esto sería más adecuado:

In [ ]:
trans = Compose(
    [
        LoadImaged(fn_keys),
        EnsureChannelFirstd(fn_keys),
        RandGaussianNoised(("img",)),
        RandSpatialCropd(fn_keys, (128, 128, 128), random_size=False),
    ]
)

ds = Dataset(filenames, trans)
item = ds[0]
im, seg = item["img"], item["seg"]
plt.imshow(np.hstack([im.numpy()[0, 48], seg.numpy()[0, 48]]))

#### DataLoader

Con el dataset definido, ya podemos crear el dataloader para crear lotes de datos:
* Hereda de la clase `DataLoader` de PyTorch con algunos cambios en los argumentos por defecto del constructor.
* La funcionalidad de MONAI es compatible con el `DataLoader` de PyTorch.
  * La funcionalidad adicional añade extensiones clave al `DataLoader` estándar.


##### Ejemplo
* El ejemplo de `DataLoader` usa cinco procesos worker para cargar los datos reales.
* MONAI también ofrece varias subclases de `Dataset` para mejorar aún más la eficiencia de este proceso.

In [ ]:
loader = DataLoader(ds, batch_size=5, num_workers=5)
batch = first(loader)
print(list(batch.keys()), batch["img"].shape)

f, ax = plt.subplots(2, 1, figsize=(8, 4))
ax[0].imshow(np.hstack(batch["img"][:, 0, 64]))
ax[1].imshow(np.hstack(batch["seg"][:, 0, 64]))

### 4. ¿Qué es un dataset de MONAI y cómo funciona el caché de datasets?


Los usuarios a menudo necesitan entrenar el modelo durante muchas épocas (potencialmente miles) sobre los datos para alcanzar la calidad deseada del modelo:
* Una implementación nativa de PyTorch puede cargar datos y ejecutar los mismos pasos de preprocesamiento en cada época durante el entrenamiento.
* Esto puede resultar lento e innecesario, especialmente cuando los volúmenes de imágenes médicas son grandes.
* Al utilizar el caché de datasets, se puede reducir el tiempo que tarda el sistema en cargar estos datos y preprocesarlos.
* Reduciendo así el tiempo total de entrenamiento.

Aquí demostraremos dos de las subclases de `Dataset`, aunque existen otras `Dataset` especializadas y útiles.



#### ¿Qué es un dataset de MONAI?

Un dataset de MONAI es un dataset genérico con una propiedad `__len__`, una propiedad `__getitem__` y una transformación de datos invocable opcional al obtener una muestra.

Comenzaremos inicializando algunos datos genéricos, llamando a la clase `Dataset` con esos datos y especificando `None` para nuestras transformaciones.

In [ ]:
items = [{"data": 4}, 
         {"data": 9}, 
         {"data": 3}, 
         {"data": 7}, 
         {"data": 1},
         {"data": 2},
         {"data": 5}]
dataset = monai.data.Dataset(items, transform=None)

print(f"Length of dataset is {len(dataset)}")
for item in dataset:
    print(item)

#### Compatibilidad de datasets con PyTorch DataLoader

Los datasets de MONAI pueden usarse con el `DataLoader` estándar de PyTorch:
* Como se mencionó, el `DataLoader` de MONAI tiene funcionalidad adicional útil.

In [ ]:
for item in torch.utils.data.DataLoader(dataset, batch_size=2):
    print(item)

#### ¿Qué es el caché de datasets y cómo se usa?

MONAI proporciona versiones multihilo de `CacheDataset` y `LMDBDataset` para acelerar las canalizaciones de transformación durante el entrenamiento:
 * coloque primero las transformaciones deterministas y después las aleatorias.
 * la salida de las transformaciones deterministas siempre es la misma, así que ¡solo se calcula una vez!
 * las transformaciones aleatorias producen resultados diferentes cada vez que se usan, por lo que deben calcularse cada vez.
 * puede proporcionar hasta 10 veces más velocidad de entrenamiento.
 

<img src="../imagenes/cache_dataset.png" style="width: 700px;"/>
 


Para demostrar el beneficio del caché de datasets, vamos a construir un dataset con una transformación lenta:
 * Para ello, vamos a llamar a la función `sleep` durante cada una de las funciones `__call__`.

In [ ]:
class SlowSquare(MapTransform):
    def __init__(self, keys):
        MapTransform.__init__(self, keys)
        print(f"keys to square it: {self.keys}")

    def __call__(self, x):
        time.sleep(1.0)
        output = {key: x[key] ** 2 for key in self.keys}
        return output

square_dataset = Dataset(items, transform=SlowSquare(keys="data"))

Como era de esperar, tardará unos 7 segundos en recorrer todos los elementos.

In [ ]:
%time for item in square_dataset: print(item)

Cada vez que ejecutamos este bucle, tarda aproximadamente 7 segundos en recorrer todos los elementos:
 * 12 minutos adicionales de tiempo de carga para 100 épocas.
 * Podemos mejorar este tiempo utilizando el caché.

#### Cache Dataset

Cuando se usa [CacheDataset](https://monai.readthedocs.io/en/stable/data.html#cachedataset), el caché se crea cuando el objeto se inicializa por primera vez, por lo que la inicialización es más lenta que la de un dataset normal:
* Al guardar en caché los resultados de las transformaciones de preprocesamiento no aleatorias, acelera la canalización de datos de entrenamiento.
* Si los datos solicitados no están en caché, todas las transformaciones se ejecutarán con normalidad.

In [ ]:
square_cached = CacheDataset(items, transform=SlowSquare(keys='data'))

Sin embargo, recuperar repetidamente los elementos de un `CacheDataset` inicializado es rápido.

In [ ]:
%timeit list(square_cached)

### Caché persistente

[PersistentDataset](https://monai.readthedocs.io/en/stable/data.html#persistentdataset) permite almacenar de forma persistente valores precalculados para gestionar de forma eficiente datos en formato de diccionario mayores que la memoria:
* De nuevo, los componentes de transformación no aleatorios se calculan cuando se usan por primera vez.
* Se almacenan en `cache_dir` para recuperarlos rápidamente en usos posteriores.

In [ ]:
square_persist = monai.data.PersistentDataset(items, transform=SlowSquare(keys='data'), cache_dir="my_cache")

%time for item in square_persist: print(item)

Durante la inicialización de `PersistentDataset` pasamos el parámetro "my_cache" como ubicación para almacenar los datos intermedios. Revisaremos ese directorio a continuación.

In [ ]:
!ls my_cache

Al consultar el dataset en las épocas siguientes, no volverá a ejecutar la transformación lenta, sino que usará los datos almacenados en caché.

In [ ]:
%timeit list(square_persist)

Las nuevas instancias de dataset pueden aprovechar los datos de caché:

In [ ]:
square_persist_1 = monai.data.PersistentDataset(items, transform=SlowSquare(keys='data'), cache_dir="my_cache")
%timeit list(square_persist_1)

#### Caché en acción

- También existe [SmartCacheDataset](https://monai.readthedocs.io/en/stable/data.html#smartcachedataset) para ocultar la latencia de las transformaciones con menor consumo de memoria.

### 5. ¿Qué datasets comunes proporciona MONAI?

Para empezar rápidamente con datos de entrenamiento populares en el dominio médico, MONAI proporciona varios datasets específicos del dominio:
* MedNISTDataset, DecathlonDataset, etc.
* Permite descargar fácilmente desde nuestro almacenamiento en AWS, extraer archivos de datos y generar elementos de entrenamiento/evaluación con transformaciones.

##### Dataset Decathlon

La función [DecathlonDataset](https://monai.readthedocs.io/en/stable/apps.html#monai.apps.CrossValidation) aprovecha las funcionalidades descritas a lo largo de este cuaderno. Estos datasets son una extensión de `CacheDataset`, que se explicó anteriormente.

In [ ]:
dataset = monai.apps.DecathlonDataset(root_dir="./", task="Task04_Hippocampus", section="training", download=True)

In [ ]:
print(dataset.get_properties("numTraining"))
print(dataset.get_properties("description"))
print(dataset[0]['image'].shape)
print(dataset[0]['label'].shape)

##### Dataset TCIA

El [Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) es un servicio que desidentifica y aloja un gran archivo público de imágenes médicas de cáncer:
* TCIA está financiado por el [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), parte del [National Cancer Institute (NCI)](https://www.cancer.gov/) de Estados Unidos, y es gestionado por el [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/).
* La función `TciaDataset` descargará y extraerá automáticamente los datasets de TCIA con segmentaciones DICOM adjuntas, y actuará como datasets de PyTorch para generar datos de entrenamiento, validación y prueba.

##### Dataset TCIA


In [ ]:
# Tomemos como ejemplo la colección "QIN-PROSTATE-Repeatability"
collection, seg_type = "QIN-PROSTATE-Repeatability", "SEG"

ds = TciaDataset(
    root_dir="./",
    collection=collection,
    section="training",
    download=False,
    download_len=1,
    seg_type=seg_type,
    progress=True,
    cache_rate=0.0,
    val_frac=0.2,
)

#print(ds.datalist[0])
#print(len(ds.datalist))

### ¿Qué redes y componentes de red proporciona MONAI y cómo se usan para crear una red?

MONAI ofrece definiciones de redes y sus componentes que heredan directamente de `torch.nn.Module`, `Sequential`, etc. Estas redes de propósito general incluyen topologías parametrizadas que se pueden ampliar fácilmente y son independientes del resto de MONAI, por lo que las redes pueden usarse con código de entrenamiento existente.

MONAI incluye los siguientes submódulos:
* `layers`: define capas de bajo nivel, así como fábricas para seleccionar capas de PyTorch y personalizadas en función de la dimensión y otros argumentos.
* `blocks`: bloques intermedios que definen conceptos reutilizables específicos a partir de los cuales se construyen las redes.
* `nets`: definiciones completas de redes para arquitecturas comunes, por ejemplo UNet, VNet, DenseNet.


Los bloques y las redes usan objetos `LayerFactory` como fábrica genérica para capas personalizadas y de PyTorch.

MONAI proporciona bloques para definir:
- convoluciones con activación y regularización
- unidades residuales
- squeeze/excitation
- downsampling/upsampling
- convoluciones subpixel

#### ¿Cómo se usan las capas de MONAI?

La funcionalidad de redes representa una oportunidad importante de diseño para MONAI:
* PyTorch es muy poco prescriptivo en cómo se definen las redes.
* Proporciona `Module` como clase base para crear una red y algunos métodos que deben implementarse.
* Aun así, no existe un patrón prescrito ni mucha funcionalidad auxiliar para inicializar redes.

La falta de funcionalidad auxiliar deja mucho margen para definir patrones de 'mejores prácticas' beneficiosos para construir nuevas redes en MONAI:
* Aunque triviales, las implementaciones de redes inflexibles son fáciles de crear.
* Ofrecemos a los usuarios un conjunto de herramientas que facilita mucho construir redes flexibles y bien diseñadas, y demostramos su valor comprometiéndonos a usarlas en las redes que construimos.

In [ ]:
from monai.networks.layers import Conv, Act, split_args, Pool

##### Convoluciones

La clase [Conv](https://monai.readthedocs.io/en/stable/networks.html#module-monai.networks.layers.Conv) tiene dos opciones para el primer argumento. El segundo argumento debe ser el número de dimensiones espaciales, `Conv[nombre, dimensión]`, por ejemplo:

In [ ]:
print(Conv[Conv.CONV, 1])
print(Conv[Conv.CONV, 2])
print(Conv[Conv.CONV, 3])
print(Conv[Conv.CONVTRANS, 1])
print(Conv[Conv.CONVTRANS, 2])
print(Conv[Conv.CONVTRANS, 3])

Las clases configuradas son las capas "vanilla" de PyTorch. Podemos crear instancias de ellas especificando los argumentos de la capa:

In [ ]:
print(Conv[Conv.CONV, 2](in_channels=1, out_channels=4, kernel_size=3))
print(Conv[Conv.CONV, 3](in_channels=1, out_channels=4, kernel_size=3))

##### Activación

Las clases [Act](https://monai.readthedocs.io/en/stable/networks.html#module-monai.networks.layers.Act) no requieren información sobre la dimensión espacial, pero sí admiten argumentos adicionales.

In [ ]:
print(Act[Act.PRELU])
Act[Act.PRELU](num_parameters=1, init=0.1)

Podrían especificarse por completo con una tupla de `(type_name, arg_dict)`, como `("prelu", {"num_parameters": 1, "init": 0.1})`:

In [ ]:
act_name, act_args = split_args(("prelu", {"num_parameters": 1, "init": 0.1}))
Act[act_name](**act_args)

#### ¿Cómo se usan estos componentes para crear una red?

##### Redes con definición flexible

Estas APIs permiten definir redes de manera flexible. A continuación crearemos una clase llamada `MyNetwork` que utiliza `Conv`, `Act` y `Pool`. Cada red requiere una función `__init__` y otra `forward`.

In [ ]:
class MyNetwork(torch.nn.Module):
    def __init__(self, dims=3, in_channels=1, out_channels=8, kernel_size=3, pool_kernel=2, act="relu"):
        super(MyNetwork, self).__init__()
        # convolution
        self.conv = Conv[Conv.CONV, dims](in_channels, out_channels, kernel_size=kernel_size)
        # activation
        act_type, act_args = split_args(act)
        self.act = Act[act_type](**act_args)
        # pooling
        self.pool = Pool[Pool.MAX, dims](pool_kernel)

    def forward(self, x: torch.Tensor):
        x = self.conv(x)
        x = self.act(x)
        x = self.pool(x)
        return x

##### Ejemplos de instanciación

Esta definición de red se puede instanciar para admitir entradas 2D o 3D, con tamaños de kernel flexibles. Resulta muy útil al adaptar el mismo diseño de arquitectura a distintas tareas, cambiando fácilmente entre 2D, 2.5D y 3D.

In [ ]:
# default network instance
default_net = MyNetwork()
print(default_net)
print(default_net(torch.ones(3, 1, 20, 20, 30)).shape)

# 2D network instance
elu_net = MyNetwork(dims=2, in_channels=3, act=("elu", {"inplace": True}))
print(elu_net)
print(elu_net(torch.ones(3, 3, 24, 24)).shape)

# 3D network instance with anisotropic kernels
sigmoid_net = MyNetwork(3, in_channels=4, kernel_size=(3, 3, 1), act="sigmoid")
print(sigmoid_net)
print(sigmoid_net(torch.ones(3, 4, 30, 30, 5)).shape)

### Redes de MONAI

MONAI proporciona más de 20 redes, entre ellas:
- UNet
- VNet
- AHNet
- regresor, clasificador, discriminador y critic tipo VGG
- HighResNet
- SENet
- UNETR

### Ejemplos de UNet

Definiremos una red UNet 2D con 2 capas ocultas que produzcan salidas con 8, 16 y 32 canales, y una capa inferior (bottleneck) que produzca salidas con 32 canales. Los valores de stride indican el stride para la convolución inicial, es decir, el downsampling en la ruta descendente y el upsampling en la ruta ascendente; además, se usarán convoluciones transpuestas para implementar el upsampling.


In [ ]:
net = monai.networks.nets.UNet(
    spatial_dims=2,  # 2 or 3 for a 2D or 3D network
    in_channels=1,  # number of input channels
    out_channels=1,  # number of output channels
    channels=[8, 16, 32],  # channel counts for layers
    strides=[2, 2]  # strides for mid layers
)

## Resumen

Hemos cubierto las transformaciones, datasets, caché y redes de MONAI. Algunos puntos clave son:

- Hay una larga lista de transformaciones específicas para medicina disponibles en MONAI.
- Existen versiones de transformaciones para arrays y para diccionarios.
- Puede crear una función lambda callable simple o una clase basada en `Transform` para crear su propia transformación personalizada.
- Puede crear un dataset de MONAI y pasarle directamente una cadena de transformaciones compuesta.
- Un dataset de MONAI es un dataset genérico con propiedades `len`, `getitem` y una transformación de datos invocable opcional al obtener una muestra.


## Resumen
- Se puede usar el caché de datasets para almacenar transformaciones y acelerar el entrenamiento. Algunas opciones incluidas son `CacheDataset`, `PersistentDataset` y `SmartCacheDataset`.
- MONAI proporciona acceso a algunos datasets de imágenes médicas muy usados, incluido `DecathlonDataset`.
- Entender las capas, bloques y redes básicas de MONAI.
- Usar las capas de MONAI para implementar una red flexible e instanciar dos ejemplos de UNet con distintos parámetros.